# Model Training
This is the actual file that trains the model based on the augmented data from the data augmentation file.

## Importing the required libraries

In [ ]:
import pandas as pd

## Importing the dataset saved by the data augmentation script

In [1]:
df = pd.read_csv('player_data_final.csv')

## Training a simple linear regression model

In [2]:
from sklearn.model_selection import train_test_split

# Split the data into a training set and a test set
df = pd.read_csv('player_data_final.csv')

# Drop the player name column
X = df[['MP', 'Min', '90s', 'G+A']]

# The target variable is the goals and assists columns
y = df[['Gls', 'Ast']]

# Store the data in the X_train, X_test, y_train, y_test variables
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [3]:
from sklearn.linear_model import LinearRegression

# Create a linear regression model
model = LinearRegression()

# Train the model on the training data
model.fit(X_train, y_train);

In [4]:
# Make predictions on the testing set
predictions = model.predict(X_test)

In [5]:
# Print the predictions
predictions_df = pd.DataFrame(predictions, columns=['Predicted_Gls', 'Predicted_Ast'])
print(predictions_df)

   Predicted_Gls  Predicted_Ast
0       2.877746       1.122254
1       0.706944       1.293056
2       2.190723       0.809277


## Calculating the Error

In [6]:
from sklearn.metrics import mean_squared_error

# Calculate the mean squared error of the model
mse = mean_squared_error(y_test, predictions)
print(f'Mean Squared Error: {mse}')

Mean Squared Error: 1.8145072667904367


## Running the model

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler

# Read your dataset
df = pd.read_csv('player_data_final.csv')

# Separate features (X) and target variables (y)
X = df[['MP', 'Min', '90s', 'Gls', 'Ast']]
y = df[['Gls', 'Ast']]

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the features
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Create and train the linear regression model
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# Hypothetical scenario: Predict after 50 matches 
hypothetical_matches = 50

# Create an empty DataFrame to store predictions for each player
predictions_df = pd.DataFrame(columns=['Player', 'Predicted_Gls', 'Predicted_Ast'])

# Iterate through each player in the dataset
for index, player_data in df.iterrows():
    # Adjust MP for the hypothetical scenario, considering a limit of 50 matches
    hypothetical_data = pd.DataFrame({
        'MP': [min(hypothetical_matches, player_data['MP'])],  # Limit to 50 matches
        'Min': [player_data['Min']],
        '90s': [player_data['90s']],
        'Gls': [player_data['Gls']],
        'Ast': [player_data['Ast']]
    })

    hypothetical_data_scaled = scaler.transform(hypothetical_data)

    predictions_hypothetical = model.predict(hypothetical_data_scaled)

    predictions_hypothetical = predictions_hypothetical.clip(min=0)

    predictions_hypothetical = predictions_hypothetical.round(2)

    predictions_hypothetical = predictions_hypothetical / player_data['MP'] * hypothetical_matches

    predictions_df = pd.concat([predictions_df, pd.DataFrame({
        'Player': [player_data['Player']],
        'Predicted_Gls': [predictions_hypothetical[0, 0]],
        'Predicted_Ast': [predictions_hypothetical[0, 1]]
    })], ignore_index=True)

# Display the predictions for each player
print(predictions_df)


                 Player Predicted_Gls Predicted_Ast
0     Federico Valverde      4.166667      8.333333
1       Jude Bellingham     45.454545      9.090909
2     Kepa Arrizabalaga           0.0           0.0
3         Dani Carvajal           5.0          10.0
4           David Alaba           0.0           5.0
5   Aurélien Tchouaméni      4.545455           0.0
6               Rodrygo      4.166667      4.166667
7     Eduardo Camavinga           0.0           0.0
8       Vinicius Júnior     11.111111      5.555556
9           Fran Garcia           0.0     11.111111
10               Joselu     20.833333      8.333333
11           Toni Kroos      4.166667          12.5
12          Luka Modrić           0.0           5.0


## Arranging the players

In [14]:
# Rounding off the predictions to the nearest integer

predictions_df['Predicted_Gls'] = predictions_df['Predicted_Gls'].astype(float).round()
predictions_df['Predicted_Ast'] = predictions_df['Predicted_Ast'].astype(float).round()

# Now arranging the players in descending order of predicted goals and assists
predictions_df['Total_Predicted'] = predictions_df['Predicted_Gls'] + predictions_df['Predicted_Ast']

predictions_df = predictions_df.sort_values(by='Total_Predicted', ascending=False)

print(predictions_df)

                 Player  Predicted_Gls  Predicted_Ast  Total_Predicted
1       Jude Bellingham           45.0            9.0             54.0
10               Joselu           21.0            8.0             29.0
8       Vinicius Júnior           11.0            6.0             17.0
11           Toni Kroos            4.0           12.0             16.0
3         Dani Carvajal            5.0           10.0             15.0
0     Federico Valverde            4.0            8.0             12.0
9           Fran Garcia            0.0           11.0             11.0
6               Rodrygo            4.0            4.0              8.0
4           David Alaba            0.0            5.0              5.0
5   Aurélien Tchouaméni            5.0            0.0              5.0
12          Luka Modrić            0.0            5.0              5.0
2     Kepa Arrizabalaga            0.0            0.0              0.0
7     Eduardo Camavinga            0.0            0.0              0.0


## Saving the predictions

In [15]:
predictions_df.to_csv('predictions.csv', index=False)